# Random Forest Model - Training & Evaluation
This notebook trains a Random Forest classifier on fuel order data to predict cancellations.

In [1]:
import sys, json, os, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib

from datetime import datetime

## 1. Load & Explore Data

In [3]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH = os.path.join(BASE_DIR, 'data', 'fuelsewa_synthetic_orders.csv')
NEW_DATA_PATH = os.path.join(BASE_DIR, 'data', 'new_order_outcomes.csv')

original = pd.read_csv(CSV_PATH)
if os.path.exists(NEW_DATA_PATH) and os.path.getsize(NEW_DATA_PATH) > 0:
    new_data = pd.read_csv(NEW_DATA_PATH)
    df = pd.concat([original, new_data], ignore_index=True)
else:
    df = original.copy()

print(f'Total samples: {len(df)}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nClass distribution:')
print(df['status'].value_counts())

Total samples: 810

Columns: ['order_id', 'userId', 'fuelType', 'quantity', 'deliveryLocation_latitude', 'deliveryLocation_longitude', 'deliveryLocation_address', 'requestSource', 'pricing_pricePerLiter', 'pricing_fuelCost', 'pricing_deliveryFee', 'pricing_emergencyFee', 'pricing_totalPrice', 'status', 'priority', 'isEmergency', 'isFarZone', 'cancelReason', 'estimatedDeliveryMinutes', 'distance_km', 'createdAt']

Class distribution:
status
delivered    554
cancelled    256
Name: count, dtype: int64


In [4]:
df.head()

,order_id,userId,fuelType,quantity,deliveryLocation_latitude,deliveryLocation_longitude,deliveryLocation_address,requestSource,pricing_pricePerLiter,pricing_fuelCost,...,pricing_emergencyFee,pricing_totalPrice,status,priority,isEmergency,isFarZone,cancelReason,estimatedDeliveryMinutes,distance_km,createdAt
0,0a6b173c7cbe43a2916f6f05,0ea0c96bdb5847ec823d6cd7,petrol,9.6,27.695811,85.289070,"Teku, Kathmandu Valley",roadside,178,1708.8,...,10,1798.8,delivered,high,True,False,NaN,24,3.65,2026-05-29T15:43:00
1,adfd887c47bc446eb2d123f2,11f21bbee0cc433eadd8677b,petrol,18.6,27.735611,85.604599,"Sundarijal, Kathmandu Valley",home,178,3310.8,...,0,4000.8,cancelled,normal,False,True,Repeated poor experiences with unknown drivers,116,34.49,2026-02-15T19:38:00
2,38a2796f97f849fe9f112b0b,c9620180f34141e8bf209b76,diesel,4.5,27.676263,85.230261,"Jamal, Kathmandu Valley",home,163,733.5,...,0,813.5,delivered,normal,False,False,NaN,15,2.98,2026-05-15T13:44:00
3,208d7eb1fa264d2e927b39b3,49c5e185c76444b7bc64a478,diesel,2.7,27.687980,85.271680,"Khumaltar, Kathmandu Valley",home,163,440.1,...,0,520.1,delivered,normal,False,False,NaN,27,1.83,2026-03-19T17:06:00
4,a2b7162b8af24b9ba443b941,fbf271d737c74f1e804aeaa9,diesel,7.0,27.672145,85.259841,"Suryabinayak, Kathmandu Valley",office,163,1141.0,...,0,1221.0,delivered,normal,False,False,NaN,10,0.30,2026-03-20T08:05:00


## 2. Preprocessing & Feature Engineering

In [5]:
df['target'] = (df['status'] == 'cancelled').astype(int)

for col in ['isEmergency', 'isFarZone']:
    df[col] = df[col].astype(float)

cat_map = {
    'fuelType': {'petrol': 0.0, 'diesel': 1.0},
    'requestSource': {'home': 0.0, 'office': 1.0, 'other': 2.0, 'roadside': 3.0},
    'priority': {'normal': 0.0, 'high': 1.0},
}
for col, mapping in cat_map.items():
    df[col] = df[col].fillna('unknown').astype(str).str.strip().map(mapping).fillna(-1.0)

def extract_hour(ts):
    return float(ts.split('T')[1].split(':')[0])

def extract_dow(ts):
    parts = ts.split('T')[0].split('-')
    y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
    return float(datetime(y, m, d).weekday())

df['hour_of_day'] = df['createdAt'].apply(extract_hour)
df['day_of_week'] = df['createdAt'].apply(extract_dow)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)

# Feature engineering
df['cost_per_km'] = np.where(
    df['distance_km'] > 0,
    df['pricing_totalPrice'] / df['distance_km'],
    0
)
df['delivery_fee_ratio'] = df['pricing_deliveryFee'] / df['pricing_totalPrice'].replace(0, 1)
df['is_night'] = ((df['hour_of_day'] < 6) | (df['hour_of_day'] > 22)).astype(float)
df['quantity_x_distance'] = df['quantity'] * df['distance_km']
df['is_peak_hour'] = (
    ((df['hour_of_day'] >= 7) & (df['hour_of_day'] < 9)) |
    ((df['hour_of_day'] >= 17) & (df['hour_of_day'] < 19))
).astype(float)
df['is_late_night'] = (df['hour_of_day'] < 5).astype(float)

df = df.sort_values(['userId', 'createdAt'])
df['past_orders'] = df.groupby('userId').cumcount().astype(int)
df['past_cancellations'] = df.groupby('userId')['target'].cumsum().shift(1).fillna(0).astype(int)
df['past_cancellation_rate'] = np.where(
    df['past_orders'] > 0,
    df['past_cancellations'] / df['past_orders'],
    0.0
)
print('Feature engineering completed (9 new features created)')

Feature engineering completed (9 new features created)


## 3. Feature Selection

In [6]:
feature_cols = [
    'quantity', 'pricing_totalPrice', 'pricing_deliveryFee',
    'pricing_emergencyFee', 'estimatedDeliveryMinutes', 'distance_km',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'isEmergency', 'isFarZone',
    'fuelType', 'requestSource', 'priority',
    'cost_per_km', 'delivery_fee_ratio', 'is_night', 'quantity_x_distance',
    'is_peak_hour', 'is_late_night',
    'past_orders', 'past_cancellations', 'past_cancellation_rate',
]


## 4. Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')
print(f'\nTraining set distribution:')
print(pd.Series(y_train).value_counts().to_string())
print(f'\nTest set distribution:')
print(pd.Series(y_test).value_counts().to_string())

NameError: name 'X' is not defined

## 5. Feature Scaling & SMOTE Oversampling

In [34]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Features scaled using StandardScaler')

smote = SMOTE(random_state=42)
X_train_s, y_train = smote.fit_resample(X_train_s, y_train)
print(f'After SMOTE - Training samples: {len(X_train_s)}, Class balance:')
print(pd.Series(y_train).value_counts().to_string())

Features scaled using StandardScaler
After SMOTE - Training samples: 886, Class balance:
1    443
0    443


## 6. Hyperparameter Tuning with GridSearchCV

In [35]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}
base_rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1)
grid = GridSearchCV(base_rf, param_grid, cv=3, scoring='f1', n_jobs=1, verbose=1)
grid.fit(X_train_s, y_train)

model = grid.best_estimator_
best_params = grid.best_params_
best_cv_score = grid.best_score_

print(f'Best CV F1 Score: {best_cv_score:.4f}')
print(f'Best parameters: {best_params}')

Fitting 3 folds for each of 108 candidates, totalling 324 fits
Best CV F1 Score: 0.8516
Best parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


## 7. Evaluation

In [ ]:
y_pred = model.predict(X_test_s)
accuracy = (y_test == y_pred).mean()
f1 = f1_score(y_test, y_pred)

print(f'Accuracy : {accuracy:.3f}')
print(f'F1 Score : {f1:.3f}')
print() 
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['delivered', 'cancelled']))

Accuracy : 0.790
F1 Score : 0.667

Classification Report:
              precision    recall  f1-score   support

   delivered       0.85      0.85      0.85       111
   cancelled       0.67      0.67      0.67        51

    accuracy                           0.79       162
   macro avg       0.76      0.76      0.76       162
weighted avg       0.79      0.79      0.79       162



## 8. Confusion Matrix

In [25]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(f'            Predicted')
print(f'            Deliv Canc')
print(f'Actual Deliv {cm[0][0]:5d} {cm[0][1]:5d}')
print(f'       Canc {cm[1][0]:5d} {cm[1][1]:5d}')

Confusion Matrix:
            Predicted
            Deliv Canc
Actual Deliv    94    17
       Canc    17    34


## 9. Feature Importance

In [26]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print('Feature Importance:')
for i in sorted_idx:
    print(f'  {feature_cols[i]:35s} {importances[i]:.4f}')

Feature Importance:
  quantity_x_distance                 0.1542
  pricing_totalPrice                  0.1228
  distance_km                         0.1075
  quantity                            0.1001
  estimatedDeliveryMinutes            0.0781
  delivery_fee_ratio                  0.0760
  cost_per_km                         0.0675
  pricing_deliveryFee                 0.0672
  isFarZone                           0.0602
  hour_of_day                         0.0466
  day_of_week                         0.0448
  requestSource                       0.0316
  fuelType                            0.0121
  is_night                            0.0086
  is_weekend                          0.0071
  priority                            0.0055
  isEmergency                         0.0052
  pricing_emergencyFee                0.0047


## 10. Test Prediction Samples

In [27]:
sample_indices = np.random.choice(len(X_test), size=min(10, len(X_test)), replace=False)
print('Sample Predictions (first 10 test samples):')
print(f'{"#":>3s} {"Actual":>8s} {"Predicted":>10s} {"Confidence":>10s}')
print('-' * 35)
for i, idx in enumerate(sample_indices):
    actual = 'cancelled' if y_test[idx] == 1 else 'delivered'
    pred = 'cancelled' if y_pred[idx] == 1 else 'delivered'
    prob = model.predict_proba(X_test_s[idx:idx+1])[0][1]
    print(f'{i:3d} {actual:>8s} {pred:>10s} {prob:>9.1%}')

Sample Predictions (first 10 test samples):
  #   Actual  Predicted Confidence
-----------------------------------
  0 delivered  delivered      1.0%
  1 delivered  delivered      1.7%
  2 delivered  cancelled     76.7%
  3 delivered  cancelled     58.7%
  4 delivered  delivered      4.7%
  5 cancelled  delivered     25.3%
  6 cancelled  cancelled     87.0%
  7 delivered  delivered     43.0%
  8 delivered  delivered     24.0%
  9 delivered  delivered     25.7%
